In [1]:
import os
import torch

print("=== 1. PYTORCH & ACCELERATOR CHECK ===")
print("PyTorch Version:", torch.__version__)
print("CUDA GPU Currently Enabled?:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}:", torch.cuda.get_device_name(i))

print("\n=== 2. CHECKING ATTACHED KAGGLE INPUT FOLDERS ===")
input_root = "/kaggle/input"
if os.path.exists(input_root):
    for item in sorted(os.listdir(input_root)):
        item_path = os.path.join(input_root, item)
        print("Found dataset folder:", item_path)
        if os.path.isdir(item_path):
            print("   Files inside:", sorted(os.listdir(item_path))[:10])
else:
    print("Not running on Kaggle! Current folder:", os.getcwd())

=== 1. PYTORCH & ACCELERATOR CHECK ===
PyTorch Version: 2.11.0+cu128
CUDA GPU Currently Enabled?: True
GPU Count: 2
GPU 0: Tesla T4
GPU 1: Tesla T4

=== 2. CHECKING ATTACHED KAGGLE INPUT FOLDERS ===
Found dataset folder: /kaggle/input/competitions
   Files inside: ['rsna-knee-abnormality-detection']


In [2]:
import os
import pandas as pd
import numpy as np

# 1. Define verified competition data path and create clean working directories (Stage 2)
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
WORK_DIR = "/kaggle/working"

for folder in ["checkpoints", "oof_predictions", "logs", "submissions"]:
    os.makedirs(os.path.join(WORK_DIR, folder), exist_ok=True)

print("Created working folders inside /kaggle/working:", sorted(os.listdir(WORK_DIR)))

# 2. Load the CSV metadata files (Stage 3.1)
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_series_df = pd.read_csv(os.path.join(DATA_DIR, "test_series.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("\n=== 1. TABLE SHAPES (ROWS, COLUMNS) ===")
print(f"train.csv            : {train_df.shape}")
print(f"train_series.csv     : {train_series_df.shape}")
print(f"test.csv             : {test_df.shape}")
print(f"test_series.csv      : {test_series_df.shape}")
print(f"sample_submission.csv: {sample_sub_df.shape}")

# 3. Verify the 12 target columns match sample_submission.csv
TARGET_COLS = list(sample_sub_df.columns[1:])
print(f"\n=== 2. TARGET COLUMNS ({len(TARGET_COLS)} TOTAL) ===")
print(TARGET_COLS)
assert list(train_df.columns[2:]) == TARGET_COLS, "Mismatch between train.csv and sample_submission.csv!"
print("Verified: train.csv and sample_submission.csv have the exact same 12 target columns in the exact same order!")

# 4. Audit labeled vs. unlabeled training studies
has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
has_any_label = train_df[TARGET_COLS].notna().any(axis=1)
gold_df = train_df[has_all_labels].copy()

print("\n=== 3. LABEL & REPORT AVAILABILITY IN TRAIN.CSV ===")
print(f"Total training studies               : {len(train_df)}")
print(f"Studies with ALL 12 expert labels    : {has_all_labels.sum()} (Gold standard set)")
print(f"Studies with PARTIAL expert labels   : {(has_any_label & ~has_all_labels).sum()}")
print(f"Studies with MISSING (NaN) labels    : {(~has_any_label).sum()}")
print(f"Studies with non-empty text Report   : {train_df['Report'].notna().sum()}")

# 5. Check class balance inside the Gold labeled studies
print("\n=== 4. CLASS DISTRIBUTION IN GOLD LABELED STUDIES ===")
stats_df = pd.DataFrame({
    "Positive_Count (1)": gold_df[TARGET_COLS].sum().astype(int),
    "Negative_Count (0)": (len(gold_df) - gold_df[TARGET_COLS].sum()).astype(int),
    "Positive_Percent (%)": (gold_df[TARGET_COLS].mean() * 100).round(1)
})
print(stats_df)

# 6. Audit MRI series metadata and check ID mapping / leakage
print("\n=== 5. MRI SERIES AUDIT & INTEGRITY CHECKS ===")
series_per_study = train_series_df.groupby("StudyInstanceUID").size()
print(f"Series per study -> Min: {series_per_study.min()}, Median: {series_per_study.median()}, Max: {series_per_study.max()}")
print("\nAnatomical Plane counts across all training series:")
print(train_series_df["Anatomical_Plane"].value_counts(dropna=False))

missing_series_studies = set(train_df["StudyInstanceUID"]) - set(train_series_df["StudyInstanceUID"])
train_test_overlap = set(train_df["StudyInstanceUID"]).intersection(set(test_df["StudyInstanceUID"]))
unique_reports = train_df["Report"].nunique()

print(f"\nTraining studies missing from train_series.csv : {len(missing_series_studies)}")
print(f"Overlap between train and test StudyInstanceUID: {len(train_test_overlap)}")
print(f"Unique text reports out of {len(train_df)} studies        : {unique_reports}")

Created working folders inside /kaggle/working: ['__notebook__.ipynb', 'checkpoints', 'logs', 'oof_predictions', 'submissions']



=== 1. TABLE SHAPES (ROWS, COLUMNS) ===
train.csv            : (4407, 14)
train_series.csv     : (24371, 5)
test.csv             : (3, 1)
test_series.csv      : (15, 5)
sample_submission.csv: (3, 13)

=== 2. TARGET COLUMNS (12 TOTAL) ===
['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 'Contusion', 'Fracture']
Verified: train.csv and sample_submission.csv have the exact same 12 target columns in the exact same order!

=== 3. LABEL & REPORT AVAILABILITY IN TRAIN.CSV ===
Total training studies               : 4407
Studies with ALL 12 expert labels    : 58 (Gold standard set)
Studies with PARTIAL expert labels   : 0
Studies with MISSING (NaN) labels    : 4349
Studies with non-empty text Report   : 4407

=== 4. CLASS DISTRIBUTION IN GOLD LABELED STUDIES ===
                  Positive_Count (1)  Negative_Count (0)  Positive_Percent (%)
ACL                               24                  34                  41.4


In [3]:
import os
import pandas as pd

# 1. Let's find where the newly added data is located in /kaggle/input/
input_root = "/kaggle/input"
print("=== ALL FOLDERS INSIDE /kaggle/input/ ===")
for folder in os.listdir(input_root):
    folder_path = os.path.join(input_root, folder)
    if os.path.isdir(folder_path):
        print(f"📁 {folder}")
        try:
            sub_files = os.listdir(folder_path)
            for sub in sub_files[:10]: # list first 10 items
                print(f"    - {sub}")
        except Exception as e:
            pass

=== ALL FOLDERS INSIDE /kaggle/input/ ===
📁 competitions
    - rsna-knee-abnormality-detection


In [4]:
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Load metadata
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

# 2. Extract the 58 Gold Standard studies (where all 12 targets are not missing/NaN)
has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
gold_df = train_df[has_all_labels].reset_index(drop=True)

print(f"Total training studies available: {len(train_df)}")
print(f"Gold Standard studies with expert labels: {len(gold_df)}")

# 3. Create a Simple PyTorch Dataset for our Gold58 studies
class GoldKneeDataset(Dataset):
    def __init__(self, df, target_cols):
        self.df = df
        self.target_cols = target_cols
        self.uids = df["StudyInstanceUID"].values
        self.labels = df[target_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        uid = self.uids[idx]
        label = self.labels[idx]
        
        # For our baseline smoke test, we return a dummy image tensor and the 12 labels
        # Later in Stage 4.2, we will hook this up to real DICOM slices!
        dummy_image = torch.zeros((3, 224, 224), dtype=torch.float32)
        target = torch.tensor(label, dtype=torch.float32)
        
        return {
            "uid": uid,
            "image": dummy_image,
            "target": target
        }

# 4. Test our Dataset class
smoke_dataset = GoldKneeDataset(gold_df, TARGET_COLS)
smoke_loader = DataLoader(smoke_dataset, batch_size=4, shuffle=True)

print(f"\nSuccessfully created Dataset with {len(smoke_dataset)} samples!")
batch = next(iter(smoke_loader))
print("Batch image tensor shape:", batch["image"].shape)  # Expected: torch.Size([4, 3, 224, 224])
print("Batch target tensor shape:", batch["target"].shape) # Expected: torch.Size([4, 12])
print("First study UID in batch:", batch["uid"][0])

Total training studies available: 4407
Gold Standard studies with expert labels: 58

Successfully created Dataset with 58 samples!


Batch image tensor shape: torch.Size([4, 3, 224, 224])
Batch target tensor shape: torch.Size([4, 12])
First study UID in batch: 1.2.826.0.1.3680043.8.498.28925345859498351203477642741908452608


In [ ]:
import os
import pandas as pd

# Search recursively inside /kaggle/input for any CSV files
print("=== SEARCHING FOR CSV FILES IN INPUTS ===")
for root, dirs, files in os.walk("/kaggle/input"):
    for file in files:
        if file.endswith(".csv"):
            full_path = os.path.join(root, file)
            print(f"Found CSV: {full_path}")

In [ ]:
import os
import pandas as pd

# 1. Load original train.csv and sample submission
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

# 2. Load one of the soft-label files from the attached notebook dataset
soft_label_path = "/kaggle/input/notebooks/evgendvorkin/rsna-versia-7/raptor_sub.csv"
soft_df = pd.read_csv(soft_label_path)

print(f"Loaded soft-label file shape: {soft_df.shape}")
print("Soft-label columns:", list(soft_df.columns[:5]), "...")

# 3. Merge soft labels into train_df based on StudyInstanceUID
# Drop original target columns from train_df if they are blank, then merge new soft labels
train_clean = train_df[['StudyInstanceUID', 'Report']].merge(soft_df, on='StudyInstanceUID', how='inner')

print(f"Merged training dataframe shape: {train_clean.shape}")
print("Missing values in merged targets:", train_clean[TARGET_COLS].isna().sum().sum())
print("First 3 rows of merged targets:\n", train_clean[['StudyInstanceUID'] + TARGET_COLS[:3]].head(3))

In [ ]:
import os
import pandas as pd

# Directly check the CSV files in Evgen Dvorkin's attached folder
folder_path = "/kaggle/input/notebooks/evgendvorkin/rsna-versia-7"
print("=== CHECKING FILES IN EVGEN DVORKIN'S FOLDER ===")
for file_name in sorted(os.listdir(folder_path)):
    if file_name.endswith(".csv"):
        full_path = os.path.join(folder_path, file_name)
        df = pd.read_csv(full_path)
        print(f"File: {file_name:<20} | Shape: {df.shape}")

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# 1. Load competition metadata and filter for the 58 Gold Standard studies
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
gold_df = train_df[has_all_labels].reset_index(drop=True)

print(f"Loaded Gold Standard dataset with {len(gold_df)} studies.")

# 2. Define our PyTorch Dataset
class GoldKneeDataset(Dataset):
    def __init__(self, df, target_cols):
        self.uids = df["StudyInstanceUID"].values
        self.labels = df[target_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.uids)
        
    def __getitem__(self, idx):
        # Dummy image tensor for baseline verification (3 channels, 224x224 pixels)
        image = torch.randn((3, 224, 224), dtype=torch.float32)
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        return image, target

# 3. Create DataLoader
dataset = GoldKneeDataset(gold_df, TARGET_COLS)
dataloader = DataLoader(dataset, batch_size=8, shuffle=True)

# 4. Define a Simple ResNet18 Baseline Model
# We modify the final classification layer to output 12 numbers (one for each abnormality)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(TARGET_COLS))

# Move model to GPU if available, otherwise CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Model successfully loaded and moved to device: {device}")

# 5. Define Loss Function (Binary Cross Entropy with Logits) and Optimizer
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 6. Run a quick 3-Epoch Training Smoke Test
model.train()
print("\n=== STARTING TRAINING SMOKE TEST ===")
for epoch in range(3):
    total_loss = 0.0
    for images, targets in dataloader:
        images = images.to(device)
        targets = targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * images.size(0)
        
    epoch_loss = total_loss / len(dataset)
    print(f"Epoch {epoch+1}/3 | Training Loss: {epoch_loss:.4f}")

print("\nSuccess! Your PyTorch training loop ran perfectly on the Gold Standard set!")

In [ ]:
import torch

# Verify GPU is now active after switching the accelerator
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Current Active Device: {device}")

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("GPU Memory Available:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2), "GB")
else:
    print("Note: Still running on CPU. Make sure Accelerator is set to GPU in the right sidebar if you want hardware acceleration!")

In [ ]:
import os
import pandas as pd
import numpy as np
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# 1. Load metadata and filter for the 58 Gold Standard studies
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
gold_df = train_df[has_all_labels].reset_index(drop=True)

# 2. Define a Real DICOM Knee Dataset class
class RealKneeDataset(Dataset):
    def __init__(self, df, series_df, data_dir, target_cols, img_size=224):
        self.df = df
        self.series_df = series_df
        self.data_dir = data_dir
        self.target_cols = target_cols
        self.img_size = img_size
        self.uids = df["StudyInstanceUID"].values
        self.labels = df[target_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.uids)
        
    def __getitem__(self, idx):
        uid = self.uids[idx]
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        # Find all series for this study
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == uid]
        
        # Pick the first available series folder
        if len(study_series) > 0:
            series_uid = study_series.iloc[0]["SeriesInstanceUID"]
            series_path = os.path.join(self.data_dir, "train_series", uid, series_uid)
            
            try:
                dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith(".dcm")])
                if len(dcm_files) > 0:
                    # Pick the middle slice of the series
                    mid_slice = dcm_files[len(dcm_files) // 2]
                    dcm_path = os.path.join(series_path, mid_slice)
                    
                    # Read DICOM pixel array
                    dcm = pydicom.dcmread(dcm_path)
                    img = dcm.pixel_array.astype(np.float32)
                    
                    # Normalize pixel values between 0 and 1
                    if img.max() > 0:
                        img = img / img.max()
                        
                    # Resize to 224x224 using simple PyTorch interpolation
                    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0) # [1, 1, H, W]
                    img_tensor = torch.nn.functional.interpolate(img_tensor, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
                    img_tensor = img_tensor.squeeze(0).repeat(3, 1, 1) # Convert grayscale [1, H, W] to 3-channel RGB [3, H, W]
                else:
                    img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
            except Exception as e:
                img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
        else:
            img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
            
        return img_tensor, target

# 3. Create DataLoader with real dataset
dataset = RealKneeDataset(gold_df, train_series_df, DATA_DIR, TARGET_COLS, img_size=224)
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=2)

print(f"Successfully initialized RealKneeDataset with {len(dataset)} samples!")

# Test fetching one batch of real images
sample_images, sample_targets = next(iter(dataloader))
print("Real batch image tensor shape:", sample_images.shape)
print("Real batch target tensor shape:", sample_targets.shape)

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Load metadata and filter for the 58 Gold Standard studies
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
gold_df = train_df[has_all_labels].reset_index(drop=True)

# 2. Reuse our RealKneeDataset class
class RealKneeDataset(Dataset):
    def __init__(self, df, series_df, data_dir, target_cols, img_size=224):
        self.df = df
        self.series_df = series_df
        self.data_dir = data_dir
        self.target_cols = target_cols
        self.img_size = img_size
        self.uids = df["StudyInstanceUID"].values
        self.labels = df[target_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.uids)
        
    def __getitem__(self, idx):
        uid = self.uids[idx]
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == uid]
        
        if len(study_series) > 0:
            series_uid = study_series.iloc[0]["SeriesInstanceUID"]
            series_path = os.path.join(self.data_dir, "train_series", uid, series_uid)
            try:
                dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith(".dcm")])
                if len(dcm_files) > 0:
                    mid_slice = dcm_files[len(dcm_files) // 2]
                    dcm = pydicom.dcmread(os.path.join(series_path, mid_slice))
                    img = dcm.pixel_array.astype(np.float32)
                    if img.max() > 0: img = img / img.max()
                    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                    img_tensor = torch.nn.functional.interpolate(img_tensor, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
                    img_tensor = img_tensor.squeeze(0).repeat(3, 1, 1)
                else:
                    img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
            except:
                img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
        else:
            img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
            
        return img_tensor, target

# 3. Create Train/Validation Split indices (80% train, 20% validation)
indices = np.arange(len(gold_df))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

full_dataset = RealKneeDataset(gold_df, train_series_df, DATA_DIR, TARGET_COLS)
train_subset = Subset(full_dataset, train_idx)
val_subset = Subset(full_dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=4, shuffle=False, num_workers=2)

print(f"Training set: {len(train_subset)} studies | Validation set: {len(val_subset)} studies")

# 4. Initialize Model, Loss, and Optimizer on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(TARGET_COLS))
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 5. Training & Local Validation Loop
print("\n=== RUNNING 2 EPOCHS OF TRAIN + VALIDATION ===")
for epoch in range(2):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    
    train_loss /= len(train_subset)
    
    # Validation Phase
    model.eval()
    val_preds = []
    val_targets = []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            val_preds.append(probs.cpu().numpy())
            val_targets.append(targets.numpy())
            
    val_preds = np.concatenate(val_preds, axis=0)
    val_targets = np.concatenate(val_targets, axis=0)
    
    # Calculate Macro ROC-AUC safely across the 12 columns
    auc_scores = []
    for i in range(len(TARGET_COLS)):
        try:
            score = roc_auc_score(val_targets[:, i], val_preds[:, i])
            auc_scores.append(score)
        except ValueError:
            # Handle case where validation subset has only 1 class for a specific label
            pass
            
    macro_auc = np.mean(auc_scores) if len(auc_scores) > 0 else 0.5
    print(f"Epoch {epoch+1}/2 | Train Loss: {train_loss:.4f} | Validation Macro ROC-AUC: {macro_auc:.4f}")

print("\nSuccess! Your validation pipeline and local metric calculation are fully operational!")

In [ ]:
import os
import pandas as pd
import numpy as np
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Load metadata and filter for the 58 Gold Standard studies
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
gold_df = train_df[has_all_labels].reset_index(drop=True)

# 2. RealKneeDataset class
class RealKneeDataset(Dataset):
    def __init__(self, df, series_df, data_dir, target_cols, img_size=224):
        self.df = df
        self.series_df = series_df
        self.data_dir = data_dir
        self.target_cols = target_cols
        self.img_size = img_size
        self.uids = df["StudyInstanceUID"].values
        self.labels = df[target_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.uids)
        
    def __getitem__(self, idx):
        uid = self.uids[idx]
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == uid]
        
        if len(study_series) > 0:
            series_uid = study_series.iloc[0]["SeriesInstanceUID"]
            series_path = os.path.join(self.data_dir, "train_series", uid, series_uid)
            try:
                dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith(".dcm")])
                if len(dcm_files) > 0:
                    mid_slice = dcm_files[len(dcm_files) // 2]
                    dcm = pydicom.dcmread(os.path.join(series_path, mid_slice))
                    img = dcm.pixel_array.astype(np.float32)
                    if img.max() > 0: img = img / img.max()
                    img_tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                    img_tensor = torch.nn.functional.interpolate(img_tensor, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
                    img_tensor = img_tensor.squeeze(0).repeat(3, 1, 1)
                else:
                    img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
            except:
                img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
        else:
            img_tensor = torch.zeros((3, self.img_size, self.img_size), dtype=torch.float32)
            
        return img_tensor, target

# 3. Split indices
indices = np.arange(len(gold_df))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

full_dataset = RealKneeDataset(gold_df, train_series_df, DATA_DIR, TARGET_COLS)
train_subset = Subset(full_dataset, train_idx)
val_subset = Subset(full_dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=4, shuffle=False, num_workers=2)

# 4. Model, Loss, Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(TARGET_COLS))
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 5. Training & Robust Validation Loop
print("\n=== RUNNING ROBUST TRAIN + VALIDATION ===")
for epoch in range(2):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    
    train_loss /= len(train_subset)
    
    # Validation Phase
    model.eval()
    val_preds = []
    val_targets = []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            val_preds.append(probs.cpu().numpy())
            val_targets.append(targets.numpy())
            
    val_preds = np.concatenate(val_preds, axis=0)
    val_targets = np.concatenate(val_targets, axis=0)
    
    # Robust AUC Calculation (skipping single-class columns)
    auc_scores = {}
    for i, col_name in enumerate(TARGET_COLS):
        y_true = val_targets[:, i]
        y_pred = val_preds[:, i]
        if len(np.unique(y_true)) > 1:
            auc_scores[col_name] = roc_auc_score(y_true, y_pred)
            
    macro_auc = np.mean(list(auc_scores.values())) if len(auc_scores) > 0 else 0.5
    print(f"Epoch {epoch+1}/2 | Train Loss: {train_loss:.4f} | Valid Macro ROC-AUC: {macro_auc:.4f} (Evaluated on {len(auc_scores)}/12 classes)")

print("\nSuccess! Validation loop completed cleanly without NaN errors!")

In [ ]:
import os
import pandas as pd
import numpy as np
import pydicom
import torch
from torch.utils.data import Dataset

class MultiPlaneKneeDataset(Dataset):
    """
    Upgraded dataset that fetches one slice from each anatomical plane 
    (Sagittal, Coronal, Axial) for a given knee study.
    """
    def __init__(self, df, series_df, data_dir, target_cols, img_size=224):
        self.df = df
        self.series_df = series_df
        self.data_dir = data_dir
        self.target_cols = target_cols
        self.img_size = img_size
        self.uids = df["StudyInstanceUID"].values
        self.labels = df[target_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.uids)
        
    def _load_slice_from_series(self, study_uid, series_uid):
        series_path = os.path.join(self.data_dir, "train_series", study_uid, series_uid)
        try:
            dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith(".dcm")])
            if len(dcm_files) > 0:
                mid_slice = dcm_files[len(dcm_files) // 2]
                dcm = pydicom.dcmread(os.path.join(series_path, mid_slice))
                img = dcm.pixel_array.astype(np.float32)
                if img.max() > 0: img = img / img.max()
                
                # Resize to target resolution
                tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                tensor = torch.nn.functional.interpolate(tensor, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
                return tensor.squeeze(0) # [1, H, W]
        except:
            pass
        return torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
        
    def __getitem__(self, idx):
        uid = self.uids[idx]
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == uid]
        
        # Find one series for each plane: Sagittal, Coronal, Axial
        planes = ["Sagittal", "Coronal", "Axial"]
        plane_tensors = []
        
        for plane in planes:
            plane_match = study_series[study_series["Anatomical_Plane"] == plane]
            if len(plane_match) > 0:
                series_uid = plane_match.iloc[0]["SeriesInstanceUID"]
                plane_tensor = self._load_slice_from_series(uid, series_uid)
            else:
                plane_tensor = torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
            plane_tensors.append(plane_tensor)
            
        # Stack planes into a 3-channel input tensor: [Channel 0: Sagittal, Channel 1: Coronal, Channel 2: Axial]
        multi_plane_image = torch.cat(plane_tensors, dim=0) # [3, H, W]
        
        return multi_plane_image, target

# Test the upgraded dataset
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])
has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
gold_df = train_df[has_all_labels].reset_index(drop=True)

multi_dataset = MultiPlaneKneeDataset(gold_df, train_series_df, DATA_DIR, TARGET_COLS, img_size=224)
sample_img, sample_tgt = multi_dataset[0]
print("Multi-plane tensor shape successfully created:", sample_img.shape) # Expected: torch.Size([3, 224, 224])

Integrating Multi-Plane Inputs into Our Training Loop

In [ ]:
import os
import pandas as pd
import numpy as np
import pydicom
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset
import torchvision.models as models
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# 1. Load metadata and filter for the 58 Gold Standard studies
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
train_series_df = pd.read_csv(os.path.join(DATA_DIR, "train_series.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

has_all_labels = train_df[TARGET_COLS].notna().all(axis=1)
gold_df = train_df[has_all_labels].reset_index(drop=True)

# 2. Upgraded MultiPlaneKneeDataset
class MultiPlaneKneeDataset(Dataset):
    def __init__(self, df, series_df, data_dir, target_cols, img_size=224):
        self.df = df
        self.series_df = series_df
        self.data_dir = data_dir
        self.target_cols = target_cols
        self.img_size = img_size
        self.uids = df["StudyInstanceUID"].values
        self.labels = df[target_cols].values.astype(np.float32)
        
    def __len__(self):
        return len(self.uids)
        
    def _load_slice_from_series(self, study_uid, series_uid):
        series_path = os.path.join(self.data_dir, "train_series", study_uid, series_uid)
        try:
            dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith(".dcm")])
            if len(dcm_files) > 0:
                mid_slice = dcm_files[len(dcm_files) // 2]
                dcm = pydicom.dcmread(os.path.join(series_path, mid_slice))
                img = dcm.pixel_array.astype(np.float32)
                if img.max() > 0: img = img / img.max()
                tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                tensor = torch.nn.functional.interpolate(tensor, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
                return tensor.squeeze(0)
        except:
            pass
        return torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
        
    def __getitem__(self, idx):
        uid = self.uids[idx]
        target = torch.tensor(self.labels[idx], dtype=torch.float32)
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == uid]
        
        planes = ["Sagittal", "Coronal", "Axial"]
        plane_tensors = []
        for plane in planes:
            plane_match = study_series[study_series["Anatomical_Plane"] == plane]
            if len(plane_match) > 0:
                series_uid = plane_match.iloc[0]["SeriesInstanceUID"]
                plane_tensor = self._load_slice_from_series(uid, series_uid)
            else:
                plane_tensor = torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
            plane_tensors.append(plane_tensor)
            
        multi_plane_image = torch.cat(plane_tensors, dim=0) # [3, H, W]
        return multi_plane_image, target

# 3. Train/Validation Split indices
indices = np.arange(len(gold_df))
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42)

full_dataset = MultiPlaneKneeDataset(gold_df, train_series_df, DATA_DIR, TARGET_COLS)
train_subset = Subset(full_dataset, train_idx)
val_subset = Subset(full_dataset, val_idx)

train_loader = DataLoader(train_subset, batch_size=4, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=4, shuffle=False, num_workers=2)

# 4. Model, Loss, Optimizer on GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, len(TARGET_COLS))
model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=1e-4)

# 5. Training & Robust Validation Loop
print("\n=== RUNNING MULTI-PLANE TRAIN + VALIDATION ===")
for epoch in range(2):
    model.train()
    train_loss = 0.0
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * images.size(0)
    
    train_loss /= len(train_subset)
    
    # Validation Phase
    model.eval()
    val_preds = []
    val_targets = []
    with torch.no_grad():
        for images, targets in val_loader:
            images = images.to(device)
            outputs = model(images)
            probs = torch.sigmoid(outputs)
            val_preds.append(probs.cpu().numpy())
            val_targets.append(targets.numpy())
            
    val_preds = np.concatenate(val_preds, axis=0)
    val_targets = np.concatenate(val_targets, axis=0)
    
    auc_scores = {}
    for i, col_name in enumerate(TARGET_COLS):
        y_true = val_targets[:, i]
        y_pred = val_preds[:, i]
        if len(np.unique(y_true)) > 1:
            auc_scores[col_name] = roc_auc_score(y_true, y_pred)
            
    macro_auc = np.mean(list(auc_scores.values())) if len(auc_scores) > 0 else 0.5
    print(f"Epoch {epoch+1}/2 | Train Loss: {train_loss:.4f} | Multi-Plane Valid AUC: {macro_auc:.4f} (Evaluated on {len(auc_scores)}/12 classes)")

print("\nSuccess! Multi-plane training and validation loop completed successfully!")

Write the Heuristic Report Parser****

In [ ]:
import os
import pandas as pd
import numpy as np

# 1. Load train.csv and sample submission
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

# 2. Define simple keyword rules for each of the 12 targets
keyword_rules = {
    'ACL': ['acl', 'anterior cruciate ligament'],
    'MCL': ['mcl', 'medial collateral ligament'],
    'Medial Meniscus': ['medial meniscus', 'medial meniscal'],
    'Lateral Meniscus': ['lateral meniscus', 'lateral meniscal'],
    'Medial OA': ['medial compartment', 'medial femorotibial', 'medial osteophytes'],
    'Lateral OA': ['lateral compartment', 'lateral femorotibial', 'lateral osteophytes'],
    'PF OA': ['patellofemoral', 'patella', 'trochlea', 'chondromalacia'],
    'Effusion': ['effusion', 'fluid'],
    'Synovitis': ['synovitis', 'synovial'],
    "Baker's": ["baker's", 'popliteal cyst'],
    'Contusion': ['contusion', 'bone bruise', 'marrow edema'],
    'Fracture': ['fracture', 'break']
}

negative_words = ['intact', 'normal', 'no tear', 'no evidence', 'preserved', 'unremarkable']

# 3. Function to parse reports and assign heuristic soft targets
def generate_heuristic_labels(df, target_cols):
    pseudo_labels = pd.DataFrame(index=df.index)
    pseudo_labels['StudyInstanceUID'] = df['StudyInstanceUID']
    
    for col in target_cols:
        keywords = keyword_rules.get(col, [col.lower()])
        scores = []
        
        for report in df['Report'].fillna(""):
            rep_lower = str(report).lower()
            # Check if any keyword appears in the report
            has_keyword = any(kw in rep_lower for kw in keywords)
            
            # Check if negative terms appear right around it
            has_negative = any(neg in rep_lower for neg in negative_words)
            
            if has_keyword and not has_negative:
                scores.append(0.8) # High probability if mentioned without negative context
            elif has_keyword:
                scores.append(0.3) # Lower probability if negated
            else:
                scores.append(0.1) # Default low probability
                
        pseudo_labels[col] = scores
        
    return pseudo_labels

# 4. Generate heuristic labels for all 4,407 studies
heuristic_labels_df = generate_heuristic_labels(train_df, TARGET_COLS)

print("=== HEURISTIC LABELS GENERATED SUCCESSFULLY ===")
print(f"Shape of heuristic labels dataframe: {heuristic_labels_df.shape}")
print("\nFirst 3 rows of generated soft labels:")
print(heuristic_labels_df[['StudyInstanceUID', 'ACL', 'Effusion', 'Fracture']].head(3))

# Save to working directory for training use
output_path = "/kaggle/working/heuristic_train_labels.csv"
heuristic_labels_df.to_csv(output_path, index=False)
print(f"\nSaved heuristic labels to: {output_path}")

Training on the Full Dataset Using Heuristic Labels

In [ ]:
import os
import pandas as pd
import torch

# 1. Load our newly generated heuristic labels
heuristic_df = pd.read_csv("/kaggle/working/heuristic_train_labels.csv")
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

print(f"Original train_df shape: {train_df.shape}")
print(f"Heuristic labels shape: {heuristic_df.shape}")

# 2. Verify alignment
merged_full_df = train_df[['StudyInstanceUID', 'Report']].merge(heuristic_df, on='StudyInstanceUID', how='inner')
print(f"Successfully merged full dataset shape: {merged_full_df.shape}")
print("Checking first row ACL soft label:", merged_full_df.iloc[0]['ACL'])

Full-Dataset Training Script

In [ ]:
import os
import pandas as pd
import numpy as np
import pydicom
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models

# 1. Load test metadata and sample submission template
DATA_DIR = "/kaggle/input/competitions/rsna-knee-abnormality-detection"
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
test_series_df = pd.read_csv(os.path.join(DATA_DIR, "test_series.csv"))
sample_sub_df = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
TARGET_COLS = list(sample_sub_df.columns[1:])

print(f"Number of test studies to predict: {len(test_df)}")

# If local test set has only 3 example rows, we follow sample_sub_df structure
# When submitted on Kaggle, Kaggle replaces test.csv with ~1,300 hidden test studies!

# 2. Test Dataset Class (using our multi-plane loader)
class TestKneeDataset(Dataset):
    def __init__(self, df, series_df, data_dir, img_size=224):
        self.df = df
        self.series_df = series_df
        self.data_dir = data_dir
        self.img_size = img_size
        self.uids = df["StudyInstanceUID"].values
        
    def __len__(self):
        return len(self.uids)
        
    def _load_slice_from_series(self, study_uid, series_uid):
        # Note: Test series folder is under test_series
        series_path = os.path.join(self.data_dir, "test_series", study_uid, series_uid)
        try:
            dcm_files = sorted([f for f in os.listdir(series_path) if f.endswith(".dcm")])
            if len(dcm_files) > 0:
                mid_slice = dcm_files[len(dcm_files) // 2]
                dcm = pydicom.dcmread(os.path.join(series_path, mid_slice))
                img = dcm.pixel_array.astype(np.float32)
                if img.max() > 0: img = img / img.max()
                tensor = torch.tensor(img, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
                tensor = torch.nn.functional.interpolate(tensor, size=(self.img_size, self.img_size), mode='bilinear', align_corners=False)
                return tensor.squeeze(0)
        except:
            pass
        return torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
        
    def __getitem__(self, idx):
        uid = self.uids[idx]
        study_series = self.series_df[self.series_df["StudyInstanceUID"] == uid]
        
        planes = ["Sagittal", "Coronal", "Axial"]
        plane_tensors = []
        for plane in planes:
            plane_match = study_series[study_series["Anatomical_Plane"] == plane]
            if len(plane_match) > 0:
                series_uid = plane_match.iloc[0]["SeriesInstanceUID"]
                plane_tensor = self._load_slice_from_series(uid, series_uid)
            else:
                plane_tensor = torch.zeros((1, self.img_size, self.img_size), dtype=torch.float32)
            plane_tensors.append(plane_tensor)
            
        multi_plane_image = torch.cat(plane_tensors, dim=0) # [3, H, W]
        return uid, multi_plane_image

# 3. Load our trained model checkpoint
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(TARGET_COLS))

# Load saved weights if available, otherwise use random init for demonstration
checkpoint_path = "/kaggle/working/checkpoints/resnet18_full_epoch1.pth"
if os.path.exists(checkpoint_path):
    model.load_state_dict(torch.load(checkpoint_path, map_location=device))
    print("Successfully loaded trained model checkpoint!")
else:
    print("Warning: Checkpoint not found. Using initialized model weights.")

model.to(device)
model.eval()

# 4. Run Inference on Test Studies
test_dataset = TestKneeDataset(test_df, test_series_df, DATA_DIR)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

predictions_list = []

print("\n=== RUNNING INFERENCE ON TEST STUDIES ===")
with torch.no_grad():
    for uids, images in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.sigmoid(outputs).cpu().numpy()[0]
        
        pred_dict = {"StudyInstanceUID": uids[0]}
        for i, col in enumerate(TARGET_COLS):
            pred_dict[col] = float(probs[i])
        predictions_list.append(pred_dict)

# 5. Create Submission DataFrame and Save
submission_df = pd.DataFrame(predictions_list)

# Ensure columns match sample_submission exact order
submission_df = submission_df[['StudyInstanceUID'] + TARGET_COLS]

# Save to required filename
sub_path = "/kaggle/working/submission.csv"
submission_df.to_csv(sub_path, index=False)

print(f"\nSuccess! Submission file saved to: {sub_path}")
print(submission_df.head())